# Navier–Stokes cylinder PINN
Demonstration of training and evaluation with PyTorch.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from types import SimpleNamespace

from train import train_and_evaluate
from models import NavierStokes2D
from utils import get_dataset, parabolic_inflow


In [ ]:
# Configure and run a short training loop
config = SimpleNamespace(
    wandb=SimpleNamespace(name='demo'),
    optim=SimpleNamespace(learning_rate=1e-3),
    training=SimpleNamespace(max_steps=1, res_batch_size=1024),
    logging=SimpleNamespace(log_every_steps=1),
    arch=SimpleNamespace(num_layers=4, hidden_dim=64),
)
workdir = '/tmp/ns_demo'
train_and_evaluate(config, workdir)


In [ ]:
# Visualize predictions from the trained model
(u_ref, v_ref, p_ref, coords, inflow_coords, outflow_coords, wall_coords, cylinder_coords, nu) = get_dataset()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
coords = coords.to(device)
Re = 1.0 / nu.item() if nu.numel() == 1 else 1.0
temporal_dom = torch.tensor([0.0, 1.0], device=device)
inflow_fn = lambda y: parabolic_inflow(y, U_max=1.5)
model = NavierStokes2D(config, inflow_fn, temporal_dom, coords, Re).to(device)
model.load_state_dict(torch.load(f'{workdir}/ckpt/{config.wandb.name}/model.pt', map_location=device))
model.eval()
with torch.no_grad():
    t = torch.tensor([temporal_dom[1]], device=device)
    u, v, p = model(t.repeat(coords.shape[0]), coords[:,0], coords[:,1])
x = coords[:,0].cpu().numpy()
y = coords[:,1].cpu().numpy()
triang = tri.Triangulation(x, y)
fig, axes = plt.subplots(3, 1, figsize=(6, 10))
axes[0].tricontourf(triang, u.cpu().numpy(), levels=100, cmap='jet')
axes[0].set_title('Predicted u')
axes[1].tricontourf(triang, v.cpu().numpy(), levels=100, cmap='jet')
axes[1].set_title('Predicted v')
axes[2].tricontourf(triang, p.cpu().numpy(), levels=100, cmap='jet')
axes[2].set_title('Predicted p')
plt.tight_layout()
plt.show()
